In [17]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from astropy.coordinates import SkyCoord
import astropy.units as u

In [24]:
# ============================================================
# Paths & device
# ============================================================

data_dir = r"C:/Raj_Stuff/Coding/VScode/Workspaces/Astro-ML/Data/XMM-Gaia-Cross-Matched Data Separate"
model_dir = r"C:/Raj_Stuff/Coding/VScode/Workspaces/Astro-ML/Models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# Load data
# ============================================================

xmm_df = pd.read_parquet(f"{data_dir}/xmm_raw.parquet")
gaia_df = pd.read_parquet(f"{data_dir}/gaia_raw.parquet")
matched_df = pd.read_parquet(f"{data_dir}/cross_matched_raw.parquet")

Using device: cuda


In [25]:
# ============================================================
# Feature columns
# ============================================================

gaia_cols = [
    "gaia_e_RA_ICRS", "gaia_e_DE_ICRS",
    "gaia_pmRA", "gaia_e_pmRA",
    "gaia_pmDE", "gaia_e_pmDE",
    "gaia_Plx", "gaia_e_Plx",
]

xmm_cols = [
    "SC_POSERR",
    "SC_DET_ML",
    "N_DETECTIONS",
    "CONFUSED",
    "HIGH_BACKGROUND",
    "No/Nx",
]

In [26]:
# In xmm_df
xmm_df["xmm_key"] = list(zip(xmm_df["DETID"], xmm_df["SRCID"]))

# In matched_df
matched_df["xmm_key"] = list(zip(matched_df["xmm_DETID"], matched_df["xmm_SRCID"]))


In [27]:
gaia_source_to_index = {
    src: idx for idx, src in gaia_df["gaia_Source"].items()
}
assert gaia_df["gaia_Source"].is_unique
assert matched_df["gaia_Source"].isin(gaia_df["gaia_Source"]).all()
assert matched_df["xmm_key"].isin(xmm_df["xmm_key"]).all()


In [28]:
xmmkey_to_gaia = dict(zip(matched_df["xmm_key"], matched_df["gaia_Source"]))

In [29]:
valid_xmm_mask = xmm_df["xmm_key"].isin(xmmkey_to_gaia)
valid_xmm_indices = xmm_df.index[valid_xmm_mask].to_numpy()


In [30]:
# ============================================================
# Load encoders
# ============================================================

from src.inference.encoder import xmm_encoder, gaia_encoder

xmm_encoder_path = f"{model_dir}/encoders/1e-3/simclr_xmm_3_frozen.pth"
gaia_encoder_path = f"{model_dir}/encoders/1e-3/simclr_gaia_3_frozen.pth"

xmm_enc = xmm_encoder(xmm_encoder_path).to(device)
gaia_enc = gaia_encoder(gaia_encoder_path).to(device)

xmm_enc.eval()
gaia_enc.eval()

for p in xmm_enc.parameters():
    p.requires_grad = False
for p in gaia_enc.parameters():
    p.requires_grad = False

In [31]:
# ============================================================
# Candidate generation
# ============================================================

def get_provisional_candidates(xmm_df, gaia_df, k_sigma=3.0):
    xmm_coords = SkyCoord(
        ra=xmm_df["SC_RA"].values * u.deg,
        dec=xmm_df["SC_DEC"].values * u.deg,
    )
    gaia_coords = SkyCoord(
        ra=gaia_df["gaia_RA_ICRS"].values * u.deg,
        dec=gaia_df["gaia_DE_ICRS"].values * u.deg,
    )

    max_radius = k_sigma * xmm_df["SC_POSERR"].max() * u.arcsec
    ix, ig, sep2d, _ = gaia_coords.search_around_sky(xmm_coords, max_radius)

    mask = sep2d.arcsec <= (k_sigma * xmm_df["SC_POSERR"].iloc[ix].values)
    return ix[mask], ig[mask], sep2d[mask].arcsec


ix, ig, seps = get_provisional_candidates(xmm_df, gaia_df)

In [32]:
# # ============================================================
# # Candidate preparation
# # ============================================================

# def prepare_candidate_matrix(xmm_idx):
#     mask = ix == xmm_idx
#     gaia_indices = ig[mask]

#     if len(gaia_indices) == 0:
#         return None

#     xmm_feat = torch.tensor(
#         xmm_df.iloc[xmm_idx][xmm_cols].values,
#         dtype=torch.float32,
#         device=device,
#     )

#     gaia_feat = torch.tensor(
#         gaia_df.iloc[gaia_indices][gaia_cols].values,
#         dtype=torch.float32,
#         device=device,
#     )

#     sep = torch.tensor(seps[mask], dtype=torch.float32, device=device)
#     poserr = xmm_df.iloc[xmm_idx]["SC_POSERR"]

#     return xmm_feat, gaia_feat, sep, poserr, gaia_indices

In [33]:
def prepare_candidate_matrix(xmm_idx):
    mask = ix == xmm_idx
    gaia_indices = ig[mask]

    if len(gaia_indices) == 0:
        return None

    # ---- XMM features (force numeric) ----
    xmm_vals = (
        xmm_df.iloc[xmm_idx][xmm_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    xmm_feat = torch.from_numpy(xmm_vals).to(device)

    # ---- Gaia features (already numeric, but be safe) ----
    gaia_vals = (
        gaia_df.iloc[gaia_indices][gaia_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    gaia_feat = torch.from_numpy(gaia_vals).to(device)

    sep = torch.tensor(seps[mask], dtype=torch.float32, device=device)
    poserr = float(xmm_df.iloc[xmm_idx]["SC_POSERR"])

    return xmm_feat, gaia_feat, sep, poserr, gaia_indices


In [34]:
print(xmm_df[xmm_cols].dtypes)


SC_POSERR          float32
SC_DET_ML          float32
N_DETECTIONS         int16
CONFUSED              bool
HIGH_BACKGROUND       bool
No/Nx              float64
dtype: object


In [35]:
# ============================================================
# Matcher model
# ============================================================

class XMMGaiaMatcher(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.02)
        self.null_bias = nn.Parameter(torch.zeros(1))

    def forward(self, z_xmm, z_gaia, sep_arcsec, xmm_poserr, alpha=1.0):
        semantic = z_gaia @ self.W @ z_xmm              # (K,)
        geom_penalty = alpha * (sep_arcsec / xmm_poserr) ** 2
        logits = semantic - geom_penalty
        return torch.cat([logits, self.null_bias])     # (K+1,)


matcher = XMMGaiaMatcher(embed_dim=64).to(device)
optimizer = Adam(matcher.parameters(), lr=1e-3)

In [ ]:
# ============================================================
# Training loop with batching
# ============================================================

def train_epoch(batch_size=32):
    matcher.train()
    total_loss = 0.0
    correct = 0
    total_valid = 0

    indices = valid_xmm_indices.copy()
    np.random.shuffle(indices)

    for start in tqdm(range(0, len(indices), batch_size), desc="Training"):
        batch_idx = indices[start : start + batch_size]

        optimizer.zero_grad()
        batch_loss = 0.0
        batch_valid = 0

        for i in batch_idx:
            prepared = prepare_candidate_matrix(i)
            if prepared is None:
                continue

            xmm_f, gaia_f, sep, err, c_indices = prepared

            with torch.no_grad():
                z_xmm = xmm_enc(xmm_f.unsqueeze(0)).squeeze(0)
                z_gaia = gaia_enc(gaia_f)

            logits = matcher(z_xmm, z_gaia, sep, err)

            xmm_key = xmm_df.iloc[i]["xmm_key"]
            
            true_gaia_source = matched_df.loc[
                matched_df["xmm_key"] == xmm_key, "gaia_Source"
            ].values

            candidate_sources = gaia_df.iloc[c_indices]["gaia_Source"].values

            match = np.where(candidate_sources == true_gaia_source)[0]
            if len(match) > 0:
                target = torch.tensor(match[0], device=device)
            else:
                target = torch.tensor(len(c_indices), device=device)  # NULL


            loss = F.cross_entropy(
                logits.unsqueeze(0), target.unsqueeze(0)
            )

            batch_loss += loss
            batch_valid += 1
            total_valid += 1

            if torch.argmax(logits) == target:
                correct += 1

        if batch_valid > 0:
            (batch_loss / batch_valid).backward()
            optimizer.step()
            total_loss += batch_loss.item()

    return (
        total_loss / max(total_valid, 1),
        correct / max(total_valid, 1),
    )

In [37]:
# ============================================================
# Run training
# ============================================================

num_epochs = 10

for epoch in range(num_epochs):
    loss, acc = train_epoch(batch_size=32)
    print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f} | Acc: {acc:.4f}")

Training: 100%|██████████| 456/456 [01:47<00:00,  4.24it/s]


Epoch 01 | Loss: 1.3268 | Acc: 0.4043


Training: 100%|██████████| 456/456 [01:39<00:00,  4.56it/s]


Epoch 02 | Loss: 0.8511 | Acc: 0.6719


Training: 100%|██████████| 456/456 [01:43<00:00,  4.40it/s]


...

Epoch 08 | Loss: 0.3087 | Acc: 0.8879


Training: 100%|██████████| 456/456 [02:14<00:00,  3.39it/s]


Epoch 09 | Loss: 0.2676 | Acc: 0.9038


Training: 100%|██████████| 456/456 [01:59<00:00,  3.81it/s]

Epoch 10 | Loss: 0.2319 | Acc: 0.9162


In [46]:
# ============================================================
# Training loop with batching
# ============================================================

def train_epoch(batch_size=32):
    matcher.train()
    total_loss = 0.0
    correct = 0
    total_valid = 0

    indices = valid_xmm_indices.copy()
    np.random.shuffle(indices)

    for start in tqdm(range(0, len(indices), batch_size), desc="Training"):
        batch_idx = indices[start : start + batch_size]

        optimizer.zero_grad()
        batch_loss = 0.0
        batch_valid = 0

        for i in batch_idx:
            prepared = prepare_candidate_matrix(i)
            if prepared is None:
                continue

            xmm_f, gaia_f, sep, err, c_indices = prepared

            with torch.no_grad():
                z_xmm = xmm_enc(xmm_f.unsqueeze(0)).squeeze(0)
                z_gaia = gaia_enc(gaia_f)
                z_gaia = torch.randn_like(z_gaia)


            logits = matcher(z_xmm, z_gaia, sep, err)

            xmm_key = xmm_df.iloc[i]["xmm_key"]
            
            # true_gaia_source = matched_df.loc[
            #     matched_df["xmm_key"] == xmm_key, "gaia_Source"
            # ].values

            candidate_sources = gaia_df.iloc[c_indices]["gaia_Source"].values

            # Use .iloc[0] to get the specific value of the first match found
            true_gaia_source_list = matched_df.loc[
                matched_df["xmm_key"] == xmm_key, "gaia_Source"
            ].values

            if len(true_gaia_source_list) == 0:
                # Handle cases where no ground truth exists if necessary
                target = torch.tensor(len(c_indices), device=device) 
            else:
                true_gaia_source = true_gaia_source_list[0] # Get the actual ID scalar
                match = np.where(candidate_sources == true_gaia_source)[0]
                # ... rest of your logic


            loss = F.cross_entropy(
                logits.unsqueeze(0), target.unsqueeze(0)
            )

            batch_loss += loss
            batch_valid += 1
            total_valid += 1

            if torch.argmax(logits) == target:
                correct += 1

        if batch_valid > 0:
            (batch_loss / batch_valid).backward()
            optimizer.step()
            total_loss += batch_loss.item()

    return (
        total_loss / max(total_valid, 1),
        correct / max(total_valid, 1),
    )

In [47]:
# ============================================================
# Run training
# ============================================================

num_epochs = 10

for epoch in range(num_epochs):
    loss, acc = train_epoch(batch_size=32)
    print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f} | Acc: {acc:.4f}")

Training:   0%|          | 0/456 [00:00<?, ?it/s]


UnboundLocalError: local variable 'target' referenced before assignment